# تمرین لورنز — حل کامل و اعتبارسنجی عددی
حل تاریخی اویلر در پوشه history حفظ شده است. این نوت‌بوک آزمون‌های همگرایی کوتاه‌مدت و طیف لیاپانوف را با معیارهای پروتکل ثبت‌شده اجرا می‌کند.
برای بازسازی تمام نمودارهای تاریخی، REBUILD_HOMEWORK=True قرار دهید؛ خروجی در پوشه جداگانه نوشته می‌شود.
مقایسه مرجع عددی با جواب تحلیلی دقیق متفاوت است؛ لیاپانوف گزارش‌شده افق محدود دارد.

In [1]:

from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, time, platform, sys
import numpy as np
import scipy
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
elif ROOT.name != 'H1_Lorenz' and (ROOT/'H1_Lorenz').exists(): ROOT = ROOT/'H1_Lorenz'
assert (ROOT/'protocols/H1_V2_frozen.md').exists(), 'Run from Homework/H1_Lorenz or its notebooks directory'
OUT=ROOT/'results'/'H1_V2'; OUT.mkdir(parents=True,exist_ok=True)
REBUILD_HOMEWORK=False
CFG={'r':[15,50],'y0':[1.,1.001],'scale':[20.,30.,60.],
     'conv_T':1.,'sample_dt':.001,'euler_steps':[.001,.0005,.00025,.000125,.0000625,.00003125,.000015625],
     'reference':[1e-11,1e-13,.01],'reference_tight':[1e-13,1e-15,.005],
     'lyap_T':600.,'burn':100.,'block':50.,'qr':.1,'rk_steps':[.01,.005,.0025],
     'gates':{'order':[.85,1.15],'accuracy':.001,'reference_abs':1e-6,'reference_frac':.01,
              'trace':.01,'lyap_refinement':.15,'r15_analytic':.03,'r50_positive':.1,'r50_zero':.1,'lyap_tail':.15}}
protocol_bytes=(ROOT/'protocols/H1_V2_frozen.md').read_bytes()
protocol_hash=hashlib.sha256(protocol_bytes).hexdigest()
config_hash=hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()
implementation_hash=hashlib.sha256((ROOT/'notebooks/Lorenz_H1_FULL_V2.ipynb').read_bytes()).hexdigest()
# Stable algorithm hash is invariant to notebook execution-output insertion.
nb=json.loads((ROOT/'notebooks/Lorenz_H1_FULL_V2.ipynb').read_text())
code_hash=hashlib.sha256(''.join(''.join(c['source']) for c in nb['cells'] if c['cell_type']=='code').encode()).hexdigest()
RUN='H1-V2-20261005'
started=datetime.now(timezone.utc).isoformat(); clock=time.monotonic()

def atomic_json(path,obj):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(obj,indent=2,allow_nan=False));json.loads(tmp.read_text());os.replace(tmp,path)

def array_hash(a):return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()

def load_unit(name):
    p=OUT/(name+'.npz'); meta=OUT/(name+'.json')
    if not p.exists() and not meta.exists():return None
    assert p.exists() and meta.exists(), f'Incomplete checkpoint {name}; reconcile before resume'
    m=json.loads(meta.read_text());assert m['config_hash']==config_hash and m['protocol_hash']==protocol_hash and m['code_hash']==code_hash
    assert hashlib.sha256(p.read_bytes()).hexdigest()==m['file_sha256']
    with np.load(p,allow_pickle=False) as z:data={k:z[k].copy() for k in z.files}
    assert set(data)==set(m['arrays'])
    for k,a in data.items():
        assert list(a.shape)==m['arrays'][k]['shape'] and np.all(np.isfinite(a)) and array_hash(a)==m['arrays'][k]['sha256']
    return data

def save_unit(name,data):
    p=OUT/(name+'.npz');tmp=OUT/(name+'.tmp.npz')
    assert all(np.all(np.isfinite(v)) for v in data.values())
    np.savez_compressed(tmp,**data)
    with np.load(tmp,allow_pickle=False) as z:
        assert all(np.array_equal(z[k],v) for k,v in data.items())
    os.replace(tmp,p)
    m={'config_hash':config_hash,'protocol_hash':protocol_hash,'code_hash':code_hash,
       'file_sha256':hashlib.sha256(p.read_bytes()).hexdigest(),
       'arrays':{k:{'shape':list(a.shape),'sha256':array_hash(a)} for k,a in data.items()}}
    atomic_json(OUT/(name+'.json'),m)
    return data

def progress(phase,done,total,unit):
    elapsed=time.monotonic()-clock
    atomic_json(OUT/'runtime_progress.json',{'run_id':RUN,'status':'RUNNING','phase':phase,
        'progress_unit':unit,'completed_units':done,'total_units':total,'started_at':started,
        'updated_at':datetime.now(timezone.utc).isoformat(),'elapsed_seconds':elapsed,
        'eta_status':'UNAVAILABLE','eta_seconds':None,'eta_reason':'phase units have heterogeneous work; tqdm shows local timing',
        'protocol_hash':protocol_hash,'config_hash':config_hash,'code_hash':code_hash})

def rhs(t,u,r):
    x,y,z=u; return np.array([10*(y-x),r*x-x*z-y,x*y-(8/3)*z])

def euler_sample(r,y0,dt):
    n=round(1/dt);stride=round(.001/dt);out=np.empty((1001,3));u=np.array([0.,y0,0.]);out[0]=u
    for i in tqdm(range(1,n+1),desc=f'Euler r={r}, dt={dt:g}',leave=False):
        u=u+dt*rhs(0,u,r)
        if i%stride==0:out[i//stride]=u
    assert np.all(np.isfinite(out));return out

print('Frozen configuration:',config_hash,'protocol:',protocol_hash)


Frozen configuration: daa8fa4ad4c7462005e8a00a6262d0b3ec221d42229eae8f2f65c81aba49f4b5 protocol: 7b460b9f5e230bab1ff5bedc0ff3f0fbf98492e6a04148e7c2334ec91c6db36c


/home/hatch/workspace/homework/H1_Lorenz/.venv/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/home/hatch/workspace/homework/H1_Lorenz/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

CONV=[];ts=np.arange(1001)*.001;scale=np.array(CFG['scale']);done=0
for r in tqdm(CFG['r'],desc='Convergence r cases'):
    for y0 in CFG['y0']:
        name=f'conv_r{r}_y{y0:g}';data=load_unit(name)
        if data is None:
            refs=[]
            for tol in [CFG['reference'],CFG['reference_tight']]:
                sol=solve_ivp(lambda t,u:rhs(t,u,r),(0,1),[0,y0,0],method='DOP853',t_eval=ts,rtol=tol[0],atol=tol[1],max_step=tol[2])
                assert sol.success and sol.y.shape==(3,1001);refs.append(sol.y.T)
            Eulers=np.stack([euler_sample(r,y0,h) for h in CFG['euler_steps']])
            # Historical-step assessment: .0001 divides the sampling grid.
            historical=euler_sample(r,y0,.0001)
            data=save_unit(name,{'t':ts,'reference':refs[0],'reference_tight':refs[1],
                                 'euler':Eulers,'historical_dt_0001':historical})
        errors=np.max(np.linalg.norm((data['euler']-data['reference_tight'][None,:,:])/scale,axis=2),axis=1)
        order=np.log2(errors[:-1]/errors[1:]);referr=np.max(np.linalg.norm((data['reference']-data['reference_tight'])/scale,axis=1))
        histerr=float(np.max(np.linalg.norm((data['historical_dt_0001']-data['reference_tight'])/scale,axis=1)))
        gates={'reference':bool(referr<min(1e-6,.01*errors[-1])),
               'decreasing':bool(np.all(np.diff(errors)<0)),
               'order':bool(np.all((order[-3:]>=.85)&(order[-3:]<=1.15))),
               'accuracy_finest':bool(errors[-1]<=.001)}
        CONV.append({'r':r,'y0':y0,'steps':CFG['euler_steps'],'errors':errors.tolist(),'orders':order.tolist(),
                     'reference_discrepancy':float(referr),'historical_dt_0001_error':histerr,
                     'historical_dt_0001_accuracy_pass':histerr<=.001,'gates':gates,'verdict':'PASS' if all(gates.values()) else 'FAIL'})
        done+=1;progress('convergence',done,4,'case')
atomic_json(OUT/'convergence.json',CONV)
fig,ax=plt.subplots(figsize=(8,5),layout='constrained')
for c in CONV:ax.loglog(c['steps'],c['errors'],'o-',label=f"r={c['r']}, y0={c['y0']}")
ax.axhline(.001,color='k',ls='--',label='Frozen accuracy budget');ax.set(xlabel='Euler dt',ylabel='Max normalized state error',title='Short-horizon verification: 0 <= t <= 1');ax.legend();fig.savefig(OUT/'convergence.png',dpi=170);plt.close(fig)
print(json.dumps(CONV,indent=2))


Convergence r cases:   0%|          | 0/2 [00:00<?, ?it/s]

Euler r=15, dt=0.001:   0%|          | 0/1000 [00:00<?, ?it/s]

Euler r=15, dt=0.0005:   0%|          | 0/2000 [00:00<?, ?it/s]

Euler r=15, dt=0.00025:   0%|          | 0/4000 [00:00<?, ?it/s]

Euler r=15, dt=0.000125:   0%|          | 0/8000 [00:00<?, ?it/s]

...

Euler r=50, dt=1.5625e-05:  67%|██████▋   | 43055/64000 [00:00<00:00, 214329.09it/s]

Euler r=50, dt=0.0001:   0%|          | 0/10000 [00:00<?, ?it/s]

Convergence r cases: 100%|██████████| 2/2 [00:03<00:00,  1.68s/it]

Convergence r cases: 100%|██████████| 2/2 [00:03<00:00,  1.68s/it]

[
  {
    "r": 15,
    "y0": 1.0,
    "steps": [
      0.001,
      0.0005,
      0.00025,
      0.000125,
      6.25e-05,
      3.125e-05,
      1.5625e-05
    ],
    "errors": [
      0.011171217512547345,
      0.005568021603461852,
      0.0027795123263075904,
      0.0013886190724893062,
      0.0006940237194872666,
      0.000346940213464384,
      0.0001734521712165559
    ],
    "orders": [
      1.0045497149098743,
      1.0023330287121217,
      1.0011808881199675,
      1.000594016490792,
      1.0002978987631252,
      1.000149171622186
    ],
    "reference_discrepancy": 5.981931116943666e-13,
    "historical_dt_0001_error": 0.0011107125322517347,
    "historical_dt_0001_accuracy_pass": false,
    "gates": {
      "reference": true,
      "decreasing": true,
      "order": true,
      "accuracy_finest": true
    },
    "verdict": "PASS"
  },
  {
    "r": 15,
    "y0": 1.001,
    "steps": [
      0.001,
      0.0005,
      0.00025,
      0.000125,
      6.25e-05,
      3.12

In [3]:

# Simultaneous fixed-step RK4 for state and 3x3 tangent matrix.
def augmented_rhs(u,V,r):
    x,y,z=u
    J=np.array([[-10.,10.,0.],[r-z,-1.,-x],[y,x,-8/3]])
    return rhs(0,u,r),J@V

def rk4(u,V,r,h):
    a,A=augmented_rhs(u,V,r)
    b,B=augmented_rhs(u+.5*h*a,V+.5*h*A,r)
    c,C=augmented_rhs(u+.5*h*b,V+.5*h*B,r)
    d,D=augmented_rhs(u+h*c,V+h*C,r)
    return u+(h/6)*(a+2*b+2*c+d),V+(h/6)*(A+2*B+2*C+D)

UNITS=[(r,1.,h,.1) for r in [15,50] for h in [.01,.005,.0025]]
UNITS += [(r,1.001,.0025,.1) for r in [15,50]]+[(r,1.,.0025,.05) for r in [15,50]]
LYAP=[]
for ui,(r,y0,h,qr) in enumerate(tqdm(UNITS,desc='Lyapunov units')):
    name=f'lyap_r{r}_y{y0:g}_h{h:g}_qr{qr:g}';data=load_unit(name)
    if data is None:
        intervals=round(600/qr);substeps=round(qr/h);u=np.array([0.,y0,0.]);V=np.eye(3)
        logs=np.empty((intervals,3));states=np.empty((intervals,3));times=np.arange(1,intervals+1)*qr
        # Segment checkpoints are resumable after interruption and preserve the same semantics.
        seg=OUT/(name+'_segment.npz');segmeta=OUT/(name+'_segment.json');start=0
        if seg.exists() or segmeta.exists():
            assert seg.exists() and segmeta.exists()
            sm=json.loads(segmeta.read_text());assert sm['config_hash']==config_hash and sm['code_hash']==code_hash and sm['protocol_hash']==protocol_hash
            assert hashlib.sha256(seg.read_bytes()).hexdigest()==sm['sha256']
            with np.load(seg,allow_pickle=False) as z:
                start=int(z['completed']);u=z['u'].copy();V=z['V'].copy();logs[:start]=z['logs'];states[:start]=z['states']
            assert 0<=start<=intervals and np.all(np.isfinite(logs[:start])) and np.allclose(V.T@V,np.eye(3),atol=1e-10)
        for k in tqdm(range(start,intervals),desc=name,leave=False):
            for _ in range(substeps):u,V=rk4(u,V,r,h)
            Q,R=np.linalg.qr(V);diag=np.diag(R);assert np.all(np.abs(diag)>0)
            signs=np.where(diag>=0,1.,-1.);V=Q*signs[None,:]
            logs[k]=np.log(np.abs(diag));states[k]=u
            if (k+1)%round(10/qr)==0 or k+1==intervals:
                assert np.all(np.isfinite(u)) and np.all(np.isfinite(logs[:k+1]))
                tmp=OUT/(name+'_segment.tmp.npz');np.savez_compressed(tmp,completed=np.array(k+1),u=u,V=V,logs=logs[:k+1],states=states[:k+1])
                with np.load(tmp,allow_pickle=False) as z:assert int(z['completed'])==k+1 and z['logs'].shape==(k+1,3)
                os.replace(tmp,seg);atomic_json(segmeta,{'config_hash':config_hash,'code_hash':code_hash,'protocol_hash':protocol_hash,'sha256':hashlib.sha256(seg.read_bytes()).hexdigest()})
                progress(name,k+1,intervals,'QR_interval')
        data=save_unit(name,{'t':times,'logs':logs,'states':states})
        seg.unlink(missing_ok=True);segmeta.unlink(missing_ok=True)
    m=data['t']>100+1e-8;L=data['logs'][m];t=data['t'][m];spectrum=np.sort(L.sum(axis=0)/500)[::-1]
    blocks=[]
    for i in range(10):blocks.append(np.sort(L[(t>100+i*50+1e-8)&(t<=150+i*50+1e-8)].sum(axis=0)/50)[::-1].tolist())
    tail100=np.sort(L[t>500+1e-8].sum(axis=0)/100)[::-1]
    tail200=np.sort(L[t>400+1e-8].sum(axis=0)/200)[::-1]
    LYAP.append({'r':r,'y0':y0,'dt':h,'qr':qr,'spectrum':spectrum.tolist(),'blocks50':blocks,
                 'tail100':tail100.tolist(),'tail200':tail200.tolist(),'trace_error':float(abs(spectrum.sum()+13.666666666666666))})
    progress('lyapunov',ui+1,len(UNITS),'integration_unit')
atomic_json(OUT/'lyapunov.json',LYAP)
print(json.dumps(LYAP,indent=2))


Lyapunov units:   0%|          | 0/10 [00:00<?, ?it/s]

lyap_r15_y1_h0.01_qr0.1:   0%|          | 0/6000 [00:00<?, ?it/s]

lyap_r15_y1_h0.01_qr0.1:   2%|▏         | 124/6000 [00:00<00:04, 1232.71it/s]

lyap_r15_y1_h0.01_qr0.1:   4%|▍         | 253/6000 [00:00<00:04, 1261.99it/s]

lyap_r15_y1_h0.01_qr0.1:   7%|▋         | 401/6000 [00:00<00:04, 1359.10it/s]

...

lyap_r50_y1_h0.0025_qr0.05:  99%|█████████▉| 11906/12000 [00:16<00:00, 603.79it/s]

lyap_r50_y1_h0.0025_qr0.05: 100%|█████████▉| 11990/12000 [00:16<00:00, 660.50it/s]

Lyapunov units: 100%|██████████| 10/10 [02:00<00:00, 14.87s/it]

Lyapunov units: 100%|██████████| 10/10 [02:00<00:00, 12.06s/it]

[
  {
    "r": 15,
    "y0": 1.0,
    "dt": 0.01,
    "qr": 0.1,
    "spectrum": [
      -0.3501401981536592,
      -0.3502453049109096,
      -12.96624642925337
    ],
    "blocks50": [
      [
        -0.3469546266575816,
        -0.353430876406988,
        -12.966246429254749
      ],
      [
        -0.3455035468892103,
        -0.35488195617535934,
        -12.966246429254744
      ],
      [
        -0.3477586047040597,
        -0.35262689836051,
        -12.966246429254747
      ],
      [
        -0.34865816660104565,
        -0.351727336463524,
        -12.966246429254745
      ],
      [
        -0.3459856023086793,
        -0.35439990075588995,
        -12.966246429254747
      ],
      [
        -0.3464991904604412,
        -0.35388631260412856,
        -12.966246429254745
      ],
      [
        -0.3497372744016336,
        -0.3506482286629356,
        -12.966246429254747
      ],
      [
        -0.3470245951235008,
        -0.3533609079410681,
        -12.96624642925474

In [4]:

def lookup(r,y0,h,qr):return next(a for a in LYAP if (a['r'],a['y0'],a['dt'],a['qr'])==(r,y0,h,qr))
G={'convergence_all':all(c['verdict']=='PASS' for c in CONV),
   'trace_all':all(a['trace_error']<=.01 for a in LYAP)}
for r in [15,50]:
    a=lookup(r,1.,.0025,.1);b=lookup(r,1.,.005,.1);q=lookup(r,1.,.0025,.05);p=lookup(r,1.001,.0025,.1)
    G[f'r{r}_dt']=abs(a['spectrum'][0]-b['spectrum'][0])<=.15
    G[f'r{r}_qr']=abs(a['spectrum'][0]-q['spectrum'][0])<=.15
    if r==15:
        expected=np.array([-.35019310244287927,-.35019310244287927,-12.966280461780894])
        G['r15_analytic']=all(np.max(np.abs(np.array(x['spectrum'])-expected))<=.03 and x['spectrum'][0]<0 for x in LYAP if x['r']==15 and x['dt']==.0025)
    else:
        G['r50_positive_zero']=all(x['spectrum'][0]>.1 and abs(x['spectrum'][1])<=.1 for x in LYAP if x['r']==50)
        G['r50_initial_condition']=abs(a['spectrum'][0]-p['spectrum'][0])<=.15
        G['r50_tail']=max(abs(a['spectrum'][0]-a[k][0]) for k in ['tail100','tail200'])<=.15
DECISION={'run_id':RUN,'gates':G,'overall':'PASS_WITHIN_FROZEN_SCOPE' if all(G.values()) else 'NOT_ALL_GATES_PASS',
          'limits':['Finite-horizon numerical evidence, not a proof of asymptotic chaos','No transition-distance or turbulence transfer','Historical trajectories remain incomplete'],
          'config_hash':config_hash,'protocol_hash':protocol_hash,'code_hash':code_hash}
atomic_json(OUT/'decision.json',DECISION)
fig,axs=plt.subplots(1,2,figsize=(12,4),layout='constrained')
for ax,r in zip(axs,[15,50]):
    a=lookup(r,1.,.0025,.1);t=np.arange(10)*50+150
    ax.plot(t,np.array(a['blocks50'])[:,0],'o-',label='50-time-unit block');ax.axhline(a['spectrum'][0],color='k',ls='--',label='500-time-unit mean');ax.axhline(0,color='gray',lw=.7);ax.set(xlabel='Block endpoint t',ylabel='Largest finite-time exponent',title=f'r={r}');ax.legend()
fig.savefig(OUT/'lyapunov.png',dpi=170);plt.close(fig)
atomic_json(OUT/'environment.json',{'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'scipy':scipy.__version__,'config':CFG})
# Display original full homework figures; optional complete historical replay in a distinct folder.
if REBUILD_HOMEWORK:
    destination=ROOT/'results'/'homework_rebuild';destination.mkdir(parents=True,exist_ok=True)
    source=(ROOT/'history'/'lorenz_homework.py').read_text()
    exec(compile(source,'historical_homework','exec'),{'__file__':str(destination/'runner.py'),'__name__':'__main__'})
files=[p for p in OUT.iterdir() if p.is_file() and p.name!='integrity.json']
atomic_json(OUT/'integrity.json',{'run_id':RUN,'code_hash':code_hash,'config_hash':config_hash,'protocol_hash':protocol_hash,
    'started_at':started,'finished_at':datetime.now(timezone.utc).isoformat(),
    'implementation_before_execution_sha256':implementation_hash,
    'files':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(files)},
    'execution':'COMPLETE','scientific_decision':DECISION['overall']})
atomic_json(OUT/'runtime_progress.json',{'run_id':RUN,'status':'COMPLETE','phase':'all','progress_unit':'test_case_or_integration_unit',
    'completed_units':14,'total_units':14,'eta_seconds':0,'eta_status':'COMPLETE','started_at':started,'finished_at':datetime.now(timezone.utc).isoformat()})
# Update final progress hash after terminal write.
receipt=json.loads((OUT/'integrity.json').read_text());receipt['files']['runtime_progress.json']=hashlib.sha256((OUT/'runtime_progress.json').read_bytes()).hexdigest();atomic_json(OUT/'integrity.json',receipt)
print(json.dumps(DECISION,indent=2))


{
  "run_id": "H1-V2-20261005",
  "gates": {
    "convergence_all": false,
    "trace_all": true,
    "r15_dt": true,
    "r15_qr": true,
    "r15_analytic": true,
    "r50_dt": true,
    "r50_qr": true,
    "r50_positive_zero": true,
    "r50_initial_condition": true,
    "r50_tail": true
  },
  "overall": "NOT_ALL_GATES_PASS",
  "limits": [
    "Finite-horizon numerical evidence, not a proof of asymptotic chaos",
    "No transition-distance or turbulence transfer",
    "Historical trajectories remain incomplete"
  ],
  "config_hash": "daa8fa4ad4c7462005e8a00a6262d0b3ec221d42229eae8f2f65c81aba49f4b5",
  "protocol_hash": "7b460b9f5e230bab1ff5bedc0ff3f0fbf98492e6a04148e7c2334ec91c6db36c",
  "code_hash": "70bfd686eb0d8931eacb485d3e49780c8e383798912d550f7656fd314a565a07"
}


## خواندن نتیجه
PASS_WITHIN_FROZEN_SCOPE یعنی معیارهای ثبت‌شده در همین دامنه پاس شده‌اند. هیچ قانون عمومی آشفتگی یا متریک پیش‌بینی گذار از آن نتیجه نمی‌شود. نمودارهای اصلی تمرین در ../history/ و نتایج جدید در ../results/H1_V2/ هستند.